# Data Science Methodology: Practical Examples

This notebook demonstrates the key phases of data science methodology through a practical example.

## Example Project: Customer Churn Prediction

We'll walk through the data science methodology using a telecommunications customer churn dataset. The goal is to predict which customers are likely to leave the service provider.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Set visualization style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Business Understanding

### Problem Definition
- A telecom company is facing high customer churn rate
- Each lost customer represents significant revenue loss
- The company wants to proactively identify customers at risk of churning

### Business Objectives
- Predict which customers are likely to churn in the near future
- Identify key factors driving customer churn
- Develop targeted retention strategies based on insights

### Success Metrics
- Build a model with at least 80% accuracy in predicting churn
- Identify top 3 factors influencing customer decisions to leave
- Enable targeting of the 20% most at-risk customers for retention campaigns

## 2. Analytic Approach

This is a binary classification problem (churn or not churn). We'll need to:
1. Use historical customer data with known churn outcomes
2. Build a supervised machine learning model
3. Use appropriate classification algorithms (Random Forest, Logistic Regression, etc.)
4. Focus on both prediction accuracy and model interpretability

## 3. Data Requirements & Collection

For a churn prediction project, we would typically need:
- Customer demographics (age, gender, location)
- Account information (contract length, payment method)
- Usage patterns (service utilization, call minutes, data usage)
- Customer service interactions (complaints, resolved issues)
- Billing information (monthly charges, total charges, payment history)

For this example, we'll simulate a telecom churn dataset:

In [ ]:
# Create a simulated telecom customer dataset
np.random.seed(42)
n = 1000  # Sample size

# Generate data
data = {
    'CustomerID': range(1, n+1),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'SeniorCitizen': np.random.choice([0, 1], n, p=[0.8, 0.2]),
    'Partner': np.random.choice(['Yes', 'No'], n),
    'Dependents': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7]),
    'Tenure': np.random.randint(1, 73, n),  # Months with the company (1-72)
    'PhoneService': np.random.choice(['Yes', 'No'], n, p=[0.9, 0.1]),
    'MultipleLines': np.random.choice(['Yes', 'No', 'No phone service'], n, p=[0.4, 0.5, 0.1]),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.4, 0.4, 0.2]),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'PaperlessBilling': np.random.choice(['Yes', 'No'], n),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'MonthlyCharges': np.random.uniform(20, 120, n),
}

# Create DataFrame
df = pd.DataFrame(data)

# Calculate total charges based on tenure and monthly charges (with some randomness)
df['TotalCharges'] = df['Tenure'] * df['MonthlyCharges'] * np.random.uniform(0.9, 1.1, n)

# Create target variable (Churn) with some realistic relationships
churn_prob = 0.2  # Base churn probability

# Factors that increase churn probability
churn_prob += np.where(df['Contract'] == 'Month-to-month', 0.3, 0)  # Month-to-month contracts have higher churn
churn_prob += np.where(df['Tenure'] < 12, 0.2, 0)  # New customers churn more
churn_prob += np.where(df['InternetService'] == 'Fiber optic', 0.1, 0)  # Fiber customers may churn more
churn_prob += np.where(df['MonthlyCharges'] > 80, 0.15, 0)  # Higher bills lead to more churn

# Factors that decrease churn probability
churn_prob -= np.where(df['Contract'] == 'Two year', 0.2, 0)  # Long contracts retain customers
churn_prob -= np.where(df['Tenure'] > 36, 0.2, 0)  # Long-time customers are more loyal
churn_prob -= np.where(df['Dependents'] == 'Yes', 0.1, 0)  # Customers with dependents tend to stay

# Ensure probabilities are valid (between 0 and 1)
churn_prob = np.clip(churn_prob, 0.01, 0.99)

# Generate churn based on probabilities
df['Churn'] = np.random.binomial(1, churn_prob)
df['Churn'] = df['Churn'].map({1: 'Yes', 0: 'No'})

# Display the first few rows
df.head()

## 4. Data Understanding

Let's explore our data to understand its characteristics and identify any patterns or issues.

In [ ]:
# Basic information about the dataset
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Summary statistics for numerical variables
print("\nSummary statistics:")
df.describe().round(2)

In [ ]:
# Distribution of categorical variables
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

categorical_cols = ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                    'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'Churn']

for i, col in enumerate(categorical_cols):
    sns.countplot(x=col, data=df, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].tick_params(axis='x', rotation=45)
    
plt.tight_layout()
plt.show()

In [ ]:
# Examine relationship between tenure and churn
plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='Tenure', data=df)
plt.title('Tenure by Churn Status')
plt.show()

# Examine relationship between monthly charges and churn
plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges by Churn Status')
plt.show()

In [ ]:
# Examine churn rate by contract type
plt.figure(figsize=(10, 6))
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0)
contract_churn_pct['Yes'].sort_values().plot(kind='bar')
plt.title('Churn Rate by Contract Type')
plt.ylabel('Churn Rate')
plt.show()

## 5. Data Preparation

Now we'll prepare our data for modeling by:
1. Handling categorical variables
2. Scaling numerical features
3. Splitting data into training and testing sets

In [ ]:
# Drop CustomerID as it's not relevant for prediction
df = df.drop('CustomerID', axis=1)

# Define features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## 6. Modeling

Let's create a Random Forest classifier model to predict customer churn:

In [ ]:
# Create a pipeline with preprocessing and modeling
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:,1]  # Probability for the positive class ("Yes")

## 7. Evaluation

Let's evaluate our model's performance:

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No', 'Yes'],
            yticklabels=['No', 'Yes'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test == 'Yes', y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

## 8. Feature Importance

Let's identify which features are most important for predicting churn:

In [ ]:
# Extract feature names from the preprocessing pipeline
preprocessor.fit(X_train)
cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = numerical_cols + cat_features.tolist()

# Get feature importances
importances = pipeline.named_steps['classifier'].feature_importances_

# Sort features by importance
indices = np.argsort(importances)[::-1]

# Plot feature importances
plt.figure(figsize=(12, 8))
plt.title("Feature Importances for Churn Prediction")
plt.bar(range(len(indices[:15])), importances[indices[:15]], align="center")
plt.xticks(range(len(indices[:15])), [feature_names[i] for i in indices[:15]], rotation=90)
plt.tight_layout()
plt.show()

print("Top 5 features impacting churn:")
for i in range(5):
    print(f"{i+1}. {feature_names[indices[i]]} (importance: {importances[indices[i]]:.4f})")

## 9. Deployment Considerations

For deploying our churn prediction model, we would need to consider:

1. **Integration with existing systems**:
   - Customer relationship management (CRM) software
   - Customer service platforms
   - Marketing automation tools

2. **Monitoring framework**:
   - Track model accuracy over time
   - Monitor for model drift
   - Set up alerts for significant performance changes

3. **Production pipeline**:
   - Automate data collection and preprocessing
   - Schedule regular model retraining
   - Implement versioning for models and data

4. **User interface**:
   - Dashboard for customer service representatives
   - Risk scoring for each customer
   - Action recommendations based on risk level

5. **Feedback loop**:
   - Track which retention strategies work for different customer segments
   - Use this data to continuously improve the model and business processes

## 10. Summary

Through this example, we've demonstrated the key phases of the data science methodology:

1. **Business Understanding**: Defined the churn prediction problem and objectives
2. **Analytic Approach**: Selected a classification approach for predicting churn
3. **Data Requirements & Collection**: Identified and obtained necessary customer data
4. **Data Understanding**: Explored the data to understand patterns and relationships
5. **Data Preparation**: Processed data for modeling (encoding, scaling, etc.)
6. **Modeling**: Built a Random Forest classifier to predict churn
7. **Evaluation**: Assessed model performance using appropriate metrics
8. **Deployment Considerations**: Outlined how to implement the model in production

This approach can be adapted to various business problems beyond churn prediction, following the same systematic methodology.